# Optimisation of Photoanode Architecture and Electrolyte Composition in Iodide-based Dye-Sensitized Solar Cells for Indoor Photovoltaics

This notebook applies **Bayesian optimisation** to identify electrolyte compositions that maximise the performance of dye-sensitised solar cells (DSSCs).

**Problem:** We have a set of experimental measurements in which five electrolyte components — I₂, LiI, BMII, TBP, and GuSCN — were varied, and three performance metrics were recorded: power conversion efficiency (PCE), open-circuit voltage (Voc), and short-circuit current density (Jsc). Running experiments is costly, so we use the existing data to *intelligently* suggest the next compositions most likely to improve performance.

**Approach:** A *Gaussian Process* (GP) is used as a surrogate model — it learns the relationship between composition and performance from the existing data and predicts outcomes at untested points. Bayesian optimisation then uses this model to balance *exploration* (trying unknown regions) and *exploitation* (refining known good regions) when proposing new experiments.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, WhiteKernel
import warnings
import matplotlib.pyplot as plt
from bayes_opt import BayesianOptimization
import seaborn as sns
import pandas as pd
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_absolute_error


import io
warnings.filterwarnings("ignore")


In [ ]:
df = pd.read_csv('./data-bo-dssc.tsv', sep='\t', decimal=',')
X = df[['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']].values
y = df['PCE(%)'].values

## 1. Exploratory Data Analysis

Before running the optimisation, we visualise the dataset to understand how the five input components relate to each other and to the performance metrics. Pairplots show scatter plots for every pair of variables, with kernel density estimates (KDE) on the diagonal — these reveal the distribution of each variable and potential correlations between them.

In [ ]:
df_plot = pd.read_csv('./data-bo-dssc.tsv', sep='\t', decimal=',')
df_plot.drop(columns=['PCE(%)', 'Voc(V)', 'Jsc(uAcm-2)'], inplace=True)


In [ ]:
print("\n" + "="*80)
print("Range of each parameter:")
param_cols = ['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']
for col in param_cols:
    print(f"{col}: {df[col].min()} - {df[col].max()}")
    

In [ ]:
param_cols = ['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']
targets = [('PCE(%)', 'pce'), ('Jsc(uAcm-2)', 'jsc'), ('Voc(V)', 'voc')]

df_plot = df[param_cols + ['Jsc(uAcm-2)']].copy()
g = sns.pairplot(df_plot, diag_kind='kde')

# Iterate over all subplots to adjust font sizes
for ax in g.axes.flatten():
    if ax:
        # Increase axis label sizes (x and y)
        ax.set_xlabel(ax.get_xlabel(), fontsize=20)
        ax.set_ylabel(ax.get_ylabel(), fontsize=20)
        
        # Increase tick label sizes (numbers on axes)
        ax.tick_params(axis='both', labelsize=16)

plt.show()

In [ ]:
# Separate pairplots for each output variable (PCE, Jsc, Voc)
param_cols = ['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']
targets = [('PCE(%)', 'pce'), ('Jsc(uAcm-2)', 'jsc'), ('Voc(V)', 'voc')]

# Create a list to store the generated images
imgs = []

for col, short in targets:
    df_plot = df[param_cols + [col]].copy()
    g = sns.pairplot(df_plot, diag_kind='kde')
    g.map_lower(sns.kdeplot, levels=4, color=".2")
    
    # Adjust size and title
    orig_size = g.fig.get_size_inches()
    g.fig.set_size_inches(orig_size * 0.75)
    #g.fig.suptitle(f'Pairplot - {col}', y=1.02)
    #g.savefig(f'pairplot_measured_data_{short}.png', dpi=300, bbox_inches='tight')
    
    



## 2. Bayesian Optimisation — PCE

We now apply Bayesian optimisation to maximise the **power conversion efficiency (PCE)**. The steps are:

1. **Define the search space** — specify physically meaningful concentration bounds for each component.
2. **Build a surrogate model** — a Gaussian Process is fitted to the existing data; it predicts both the expected performance and its uncertainty at any untested point.
3. **Register prior observations** — all existing experiments are fed to the optimiser as their starting knowledge.
4. **Run the optimisation loop** — in each iteration, the optimiser uses an *acquisition function* to select the next candidate point, balancing high predicted performance with high uncertainty.
5. **Extract candidates** — configurations whose predicted PCE exceeds the current experimental maximum are retained as recommendations for future experiments.

In [ ]:
bounds = {'I2': (0, 0.05),
'LiI': (0, 0.5),
'BMII': (0, 1.5),
'TBP': (0, 2),
'GuSCN': (0, 0.5)}

In [ ]:
# Verify the current maximum value of PCE
current_max = y.max()
print(f"Current maximum value of PCE(%): {current_max:.4f}")
print(f"Configuration with maximum PCE:")
max_idx = y.argmax()
for i, param in enumerate(['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']):
    print(f"  {param}: {X[max_idx, i]:.4f}")

In [ ]:
# Define the objective function for Bayesian optimisation
def black_box_function(I2, LiI, BMII, TBP, GuSCN):
    """
    Function that the optimiser will attempt to maximise.
    Employs a Gaussian Process fitted to the existing data.
    """
    # Create the point to be evaluated
    x_test = np.array([[I2, LiI, BMII, TBP, GuSCN]])
    
    # Fit the GP using all available data
    kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y)
    
    # Predict the value
    y_pred, sigma = gp.predict(x_test, return_std=True)
    
    return y_pred[0]

In [ ]:
# Initialise the Bayesian optimiser
optimizer = BayesianOptimization(
    f=black_box_function,
    pbounds=bounds,
    random_state=42,
    verbose=2
)

# Register the existing data as initial observations
for i in range(len(X)):
    optimizer.register(
        params={
            'I2': X[i, 0],
            'LiI': X[i, 1],
            'BMII': X[i, 2],
            'TBP': X[i, 3],
            'GuSCN': X[i, 4]
        },
        target=y[i]
    )

print(f"\nInitial data registered: {len(X)} experiments")

In [ ]:
# Run Bayesian optimisation iterations to identify novel candidate configurations
optimizer.maximize(
    init_points=0,  # Initial observations already registered
    n_iter=100,       # 100 new candidate configurations
)

In [ ]:
# Extract proposed configurations that surpass the current maximum
print("\n" + "="*80)
print("PROPOSED CONFIGURATIONS FOR IMPROVING PCE(%)")
print("="*80)
current_max = y.max()
print(f"Current maximum in the dataset: {current_max:.4f}%")
print("="*80)

proposed_combinations = []

# Retrieve only the last 100 iterations (the new proposals)
all_iterations = len(optimizer.res)
start_idx = len(X)  # Start after the original data

# Filter only the configurations that surpass the current maximum
for res in optimizer.res[start_idx:]:
    params = res['params']
    target = res['target']
    
    # Only include if the predicted PCE exceeds the current maximum
    if target > current_max:
        combination = {
            'I2': params['I2'],
            'LiI': params['LiI'],
            'BMII': params['BMII'],
            'TBP': params['TBP'],
            'GuSCN': params['GuSCN'],
            'PCE_predicted(%)': target,
            'Improvement(%)': target - current_max
        }
        proposed_combinations.append(combination)

# Sort by predicted PCE (descending) and retain the top results
proposed_combinations = sorted(proposed_combinations, key=lambda x: x['PCE_predicted(%)'], reverse=True)

if proposed_combinations:
    print(f"\n{len(proposed_combinations)} configurations found that surpass the current maximum:\n")
    for idx, combination in enumerate(proposed_combinations, 1):
        print(f"Experiment {idx}:")
        print(f"  I2:     {combination['I2']:.4f}")
        print(f"  LiI:    {combination['LiI']:.4f}")
        print(f"  BMII:   {combination['BMII']:.4f}")
        print(f"  TBP:    {combination['TBP']:.4f}")
        print(f"  GuSCN:  {combination['GuSCN']:.4f}")
        print(f"  Predicted PCE: {combination['PCE_predicted(%)']:.4f}%")
        print(f"  Improvement: +{combination['Improvement(%)']:.4f}%\n")
    
    print("="*80)
    print(f"Best result found: {proposed_combinations[0]['PCE_predicted(%)']:.4f}%")
    print(f"Improvement over current maximum: +{proposed_combinations[0]['Improvement(%)']:.4f}%")
    print("="*80)
else:
    print("\n⚠ No configurations found that surpass the current maximum.")
    print("This may indicate that the model regards the current maximum as already optimal.")
    print("="*80)

In [ ]:
# Create a DataFrame from the proposals only if there are configurations that improve upon the current maximum
if proposed_combinations:
    df_proposed = pd.DataFrame(proposed_combinations)
    # Add experiment number
    df_proposed.insert(0, 'Experiment', range(1, len(proposed_combinations) + 1))
    
    print("\nTable of proposed experiments (only those that surpass the current maximum):")
    print(df_proposed.to_string(index=False))
    
    # Save the proposals to a CSV file
    output_file = './proposed_bo_experiments_pce.csv'
    df_proposed.to_csv(output_file, index=False)
    print(f"\n✓ {len(proposed_combinations)} proposals saved to: {output_file}")
else:
    print("\n⚠ No experiments surpass the current maximum; nothing to save.")

In [ ]:
# Visualisation of the optimisation progression
if proposed_combinations:
    plt.figure(figsize=(12, 5))

    # Subplot 1: Evolution of the maximum found
    plt.subplot(1, 2, 1)
    max_values = [optimizer.res[i]['target'] for i in range(len(X), len(optimizer.res))]
    cummax_values = np.maximum.accumulate(max_values)
    plt.plot(range(1, len(max_values) + 1), max_values, 'bo-', alpha=0.5, markersize=3, label='Predicted value')
    plt.plot(range(1, len(max_values) + 1), cummax_values, 'r--', linewidth=2, label='Cumulative maximum')
    plt.axhline(y=current_max, color='g', linestyle=':', linewidth=2, label=f'Current maximum ({current_max:.4f}%)')
    plt.xlabel('Iteration')
    plt.ylabel('PCE(%)')
    plt.title('Bayesian Optimisation Progression')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Subplot 2: Parameter comparison
    plt.subplot(1, 2, 2)
    param_names = ['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']
    best_config = proposed_combinations[0]  # Best configuration found
    current_best_params = X[max_idx]

    x_pos = np.arange(len(param_names))
    width = 0.35

    plt.bar(x_pos - width/2, current_best_params, width, label='Current best', alpha=0.7, color='green')
    plt.bar(x_pos + width/2, [best_config[p] for p in param_names], width, label='Predicted best', alpha=0.7, color='orange')
    plt.xlabel('Parameters')
    plt.ylabel('Value')
    plt.title('Configuration Comparison')
    plt.xticks(x_pos, param_names)
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    #plt.savefig('./bayesian_optimisation_results.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("✓ Figures saved to: ./bayesian_optimisation_results_pce.png")
else:
    print("⚠ No figures generated as no configurations improve upon the current maximum.")

## 3. Bayesian Optimisation — $V_{oc}$

The same procedure is repeated, this time targeting the **open-circuit voltage ($V_{oc}$)**. A new Gaussian Process surrogate is fitted to the $V_{oc}$ measurements, and the optimiser searches for electrolyte compositions predicted to yield voltages higher than the current experimental maximum.

In [ ]:
df = pd.read_csv('./data-bo-dssc.tsv', sep='\t', decimal=',')
X = df[['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']].values
y = df['Voc(V)'].values

In [ ]:
bounds = {'I2': (0, 0.05),
'LiI': (0, 0.5),
'BMII': (0, 1.5),
'TBP': (0, 2),
'GuSCN': (0, 0.5)}

In [ ]:
# Verify the current maximum value of Voc
current_max = y.max()
print(f"Current maximum value of Voc(V): {current_max:.4f}")
print(f"Configuration with maximum Voc:")
max_idx = y.argmax()
for i, param in enumerate(['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']):
    print(f"  {param}: {X[max_idx, i]:.4f}")

In [ ]:
# Define the objective function for Bayesian optimisation
def black_box_function(I2, LiI, BMII, TBP, GuSCN):
    """
    Function that the optimiser will attempt to maximise.
    Employs a Gaussian Process fitted to the existing data.
    """
    # Create the point to be evaluated
    x_test = np.array([[I2, LiI, BMII, TBP, GuSCN]])
    
    # Fit the GP using all available data
    kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y)
    
    # Predict the value
    y_pred, sigma = gp.predict(x_test, return_std=True)
    
    return y_pred[0]

In [ ]:
# Initialise the Bayesian optimiser
optimizer = BayesianOptimization(
    f=black_box_function,
    pbounds=bounds,
    random_state=42,
    verbose=2
)

# Register the existing data as initial observations
for i in range(len(X)):
    optimizer.register(
        params={
            'I2': X[i, 0],
            'LiI': X[i, 1],
            'BMII': X[i, 2],
            'TBP': X[i, 3],
            'GuSCN': X[i, 4]
        },
        target=y[i]
    )

print(f"\nInitial data registered: {len(X)} experiments")

In [ ]:
# Run Bayesian optimisation iterations to identify novel candidate configurations
optimizer.maximize(
    init_points=0,  # Initial observations already registered
    n_iter=100,       # 100 new candidate configurations
)

In [ ]:
# Extract proposed configurations that surpass the current maximum
print("\n" + "="*80)
print("PROPOSED CONFIGURATIONS FOR IMPROVING Voc")
print("="*80)
current_max = y.max()
print(f"Current maximum in the dataset: {current_max:.4f} V")
print("="*80)

proposed_combinations = []

# Retrieve only the last 100 iterations (the new proposals)
all_iterations = len(optimizer.res)
start_idx = len(X)  # Start after the original data

# Filter only the configurations that surpass the current maximum
for res in optimizer.res[start_idx:]:
    params = res['params']
    target = res['target']
    
    # Only include if the predicted Voc exceeds the current maximum
    if target > current_max:
        combination = {
            'I2': params['I2'],
            'LiI': params['LiI'],
            'BMII': params['BMII'],
            'TBP': params['TBP'],
            'GuSCN': params['GuSCN'],
            'Voc_predicted(V)': target,
            'Improvement(V)': target - current_max
        }
        proposed_combinations.append(combination)

# Sort by predicted Voc (descending) and retain the top results
proposed_combinations = sorted(proposed_combinations, key=lambda x: x['Voc_predicted(V)'], reverse=True)

if proposed_combinations:
    print(f"\n{len(proposed_combinations)} configurations found that surpass the current maximum:\n")
    for idx, combination in enumerate(proposed_combinations, 1):
        print(f"Experiment {idx}:")
        print(f"  I2:     {combination['I2']:.4f}")
        print(f"  LiI:    {combination['LiI']:.4f}")
        print(f"  BMII:   {combination['BMII']:.4f}")
        print(f"  TBP:    {combination['TBP']:.4f}")
        print(f"  GuSCN:  {combination['GuSCN']:.4f}")
        print(f"  Predicted Voc: {combination['Voc_predicted(V)']:.4f} V")
        print(f"  Improvement: +{combination['Improvement(V)']:.4f} V\n")
    
    print("="*80)
    print(f"Best result found: {proposed_combinations[0]['Voc_predicted(V)']:.4f} V")
    print(f"Improvement over current maximum: +{proposed_combinations[0]['Improvement(V)']:.4f} V")
    print("="*80)
else:
    print("\n⚠ No configurations found that surpass the current maximum.")
    print("This may indicate that the model regards the current maximum as already optimal.")
    print("="*80)

In [ ]:
# Create a DataFrame from the proposals only if there are configurations that improve upon the current maximum
if proposed_combinations:
    df_proposed = pd.DataFrame(proposed_combinations)
    # Add experiment number
    df_proposed.insert(0, 'Experiment', range(1, len(proposed_combinations) + 1))
    
    print("\nTable of proposed experiments (only those that surpass the current maximum):")
    print(df_proposed.to_string(index=False))
    
    # Save the proposals to a CSV file
    output_file = './proposed_bo_experiments_voc.csv'
    df_proposed.to_csv(output_file, index=False)
    print(f"\n✓ {len(proposed_combinations)} proposals saved to: {output_file}")
else:
    print("\n⚠ No experiments surpass the current maximum; nothing to save.")

In [ ]:
# Visualisation of the optimisation progression
if proposed_combinations:
    plt.figure(figsize=(12, 5))

    # Subplot 1: Evolution of the maximum found
    plt.subplot(1, 2, 1)
    max_values = [optimizer.res[i]['target'] for i in range(len(X), len(optimizer.res))]
    cummax_values = np.maximum.accumulate(max_values)
    plt.plot(range(1, len(max_values) + 1), max_values, 'bo-', alpha=0.5, markersize=3, label='Predicted value')
    plt.plot(range(1, len(max_values) + 1), cummax_values, 'r--', linewidth=2, label='Cumulative maximum')
    plt.axhline(y=current_max, color='g', linestyle=':', linewidth=2, label=f'Current maximum ({current_max:.4f} V)')
    plt.xlabel('Iteration')
    plt.ylabel('Voc(V)')
    plt.title('Bayesian Optimisation Progression')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Subplot 2: Parameter comparison
    plt.subplot(1, 2, 2)
    param_names = ['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']
    best_config = proposed_combinations[0]  # Best configuration found
    current_best_params = X[max_idx]

    x_pos = np.arange(len(param_names))
    width = 0.35

    plt.bar(x_pos - width/2, current_best_params, width, label='Current best', alpha=0.7, color='green')
    plt.bar(x_pos + width/2, [best_config[p] for p in param_names], width, label='Predicted best', alpha=0.7, color='orange')
    plt.xlabel('Parameters')
    plt.ylabel('Value')
    plt.title('Configuration Comparison')
    plt.xticks(x_pos, param_names)
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    #plt.savefig('./bayesian_optimisation_results_voc.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("✓ Figures saved to: ./bayesian_optimisation_results_voc.png")
else:
    print("⚠ No figures generated as no configurations improve upon the current maximum.")

## 4. Bayesian Optimisation — $J_{sc}$

Finally, the optimisation is repeated for the **short-circuit current density ($J_{sc}$)**. This completes the three-target analysis, giving a set of candidate compositions that may independently improve each performance metric.

In [ ]:
df = pd.read_csv('./data-bo-dssc.tsv', sep='\t', decimal=',')
X = df[['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']].values
y = df['Jsc(uAcm-2)'].values

In [ ]:
bounds = {'I2': (0, 0.05),
'LiI': (0, 0.5),
'BMII': (0, 1.5),
'TBP': (0, 2),
'GuSCN': (0, 0.5)}

In [ ]:
# Verify the current maximum value of Jsc
current_max = y.max()
print(f"Current maximum value of Jsc(uAcm-2): {current_max:.4f}")
print(f"Configuration with maximum Jsc(uAcm-2):")
max_idx = y.argmax()
for i, param in enumerate(['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']):
    print(f"  {param}: {X[max_idx, i]:.4f}")

In [ ]:
# Define the objective function for Bayesian optimisation
def black_box_function(I2, LiI, BMII, TBP, GuSCN):
    """
    Function that the optimiser will attempt to maximise.
    Employs a Gaussian Process fitted to the existing data.
    """
    # Create the point to be evaluated
    x_test = np.array([[I2, LiI, BMII, TBP, GuSCN]])
    
    # Fit the GP using all available data
    kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y)
    
    # Predict the value
    y_pred, sigma = gp.predict(x_test, return_std=True)
    
    return y_pred[0]

In [ ]:
# Initialise the Bayesian optimiser
optimizer = BayesianOptimization(
    f=black_box_function,
    pbounds=bounds,
    random_state=42,
    verbose=2
)

# Register the existing data as initial observations
for i in range(len(X)):
    optimizer.register(
        params={
            'I2': X[i, 0],
            'LiI': X[i, 1],
            'BMII': X[i, 2],
            'TBP': X[i, 3],
            'GuSCN': X[i, 4]
        },
        target=y[i]
    )

print(f"\nInitial data registered: {len(X)} experiments")

In [ ]:
# Run Bayesian optimisation iterations to identify novel candidate configurations
optimizer.maximize(
    init_points=0,  # Initial observations already registered
    n_iter=100,       # 100 new candidate configurations
)

In [ ]:
# Extract proposed configurations that surpass the current maximum
print("\n" + "="*80)
print("PROPOSED CONFIGURATIONS FOR IMPROVING Jsc")
print("="*80)
current_max = y.max()
print(f"Current maximum in the dataset: {current_max:.4f} uAcm-2")
print("="*80)

proposed_combinations = []

# Retrieve only the last 100 iterations (the new proposals)
all_iterations = len(optimizer.res)
start_idx = len(X)  # Start after the original data

# Filter only the configurations that surpass the current maximum
for res in optimizer.res[start_idx:]:
    params = res['params']
    target = res['target']
    
    # Only include if the predicted Jsc exceeds the current maximum
    if target > current_max:
        combination = {
            'I2': params['I2'],
            'LiI': params['LiI'],
            'BMII': params['BMII'],
            'TBP': params['TBP'],
            'GuSCN': params['GuSCN'],
            'Jsc_predicted(uAcm-2)': target,
            'Improvement(uAcm-2)': target - current_max
        }
        proposed_combinations.append(combination)

# Sort by predicted Jsc (descending) and retain the top results
proposed_combinations = sorted(proposed_combinations, key=lambda x: x['Jsc_predicted(uAcm-2)'], reverse=True)

if proposed_combinations:
    print(f"\n{len(proposed_combinations)} configurations found that surpass the current maximum:\n")
    for idx, combination in enumerate(proposed_combinations, 1):
        print(f"Experiment {idx}:")
        print(f"  I2:     {combination['I2']:.4f}")
        print(f"  LiI:    {combination['LiI']:.4f}")
        print(f"  BMII:   {combination['BMII']:.4f}")
        print(f"  TBP:    {combination['TBP']:.4f}")
        print(f"  GuSCN:  {combination['GuSCN']:.4f}")
        print(f"  Predicted Jsc: {combination['Jsc_predicted(uAcm-2)']:.4f} uAcm-2")
        print(f"  Improvement: +{combination['Improvement(uAcm-2)']:.4f} uAcm-2\n")
    
    print("="*80)
    print(f"Best result found: {proposed_combinations[0]['Jsc_predicted(uAcm-2)']:.4f} uAcm-2")
    print(f"Improvement over current maximum: +{proposed_combinations[0]['Improvement(uAcm-2)']:.4f} uAcm-2")
    print("="*80)
else:
    print("\n⚠ No configurations found that surpass the current maximum.")
    print("This may indicate that the model regards the current maximum as already optimal.")
    print("="*80)

In [ ]:
# Create a DataFrame from the proposals only if there are configurations that improve upon the current maximum
if proposed_combinations:
    df_proposed = pd.DataFrame(proposed_combinations)
    # Add experiment number
    df_proposed.insert(0, 'Experiment', range(1, len(proposed_combinations) + 1))
    
    print("\nTable of proposed experiments (only those that surpass the current maximum):")
    print(df_proposed.to_string(index=False))
    
    # Save the proposals to a CSV file
    output_file = './proposed_bo_experiments_jsc.csv'
    df_proposed.to_csv(output_file, index=False)
    print(f"\n✓ {len(proposed_combinations)} proposals saved to: {output_file}")
else:
    print("\n⚠ No experiments surpass the current maximum; nothing to save.")

In [ ]:
# Visualisation of the optimisation progression
if proposed_combinations:
    fig = plt.figure(figsize=(16, 10))
    
    # Create grid: top row for progression, bottom row for 5 individual parameters sharing the y-axis
    gs = fig.add_gridspec(2, 5, height_ratios=[1.2, 1], hspace=0.35, wspace=0.05)
    
    # Upper subplot: Evolution of the maximum found (spans the entire top row)
    ax_evolution = fig.add_subplot(gs[0, :])
    max_values = [optimizer.res[i]['target'] for i in range(len(X), len(optimizer.res))]
    cummax_values = np.maximum.accumulate(max_values)
    ax_evolution.plot(range(1, len(max_values) + 1), max_values, 'bo-', alpha=0.5, markersize=3, label='Predicted value')
    ax_evolution.plot(range(1, len(max_values) + 1), cummax_values, 'r--', linewidth=2, label='Cumulative maximum')
    ax_evolution.axhline(y=current_max, color='g', linestyle=':', linewidth=2, label=f'Current maximum ({current_max:.4f})')
    ax_evolution.set_xlabel('Iteration', fontsize=12)
    ax_evolution.set_ylabel('Jsc(uAcm-2)', fontsize=12)
    ax_evolution.set_title('Bayesian Optimisation Progression', fontsize=14, fontweight='bold')
    ax_evolution.legend(fontsize=10)
    ax_evolution.grid(True, alpha=0.3)

    # Lower subplots: Individual comparison of each parameter (5 plots sharing the y-axis)
    param_names = ['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']
    best_config = proposed_combinations[0]  # Best configuration found
    current_best_params = X[max_idx]
    
    # Determine the shared y-axis range
    all_values = []
    for i, param in enumerate(param_names):
        all_values.extend([current_best_params[i], best_config[param]])
    y_max = max(all_values) * 1.15  # 15% upper margin
    
    for i, param in enumerate(param_names):
        if i == 0:
            ax = fig.add_subplot(gs[1, i])
            ax_first = ax
        else:
            ax = fig.add_subplot(gs[1, i], sharey=ax_first)
        
        x_pos = np.array([0, 1])
        values = [current_best_params[i], best_config[param]]
        colors = ['green', 'orange']
        
        bars = ax.bar(x_pos, values, color=colors, alpha=0.7, width=0.6)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(['Current', 'Predicted'], fontsize=9, rotation=45, ha='right')
        ax.set_title(f'{param}', fontsize=11, fontweight='bold')
        ax.set_ylim(0, y_max)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Only show the y-axis label on the first subplot
        if i == 0:
            ax.set_ylabel('Concentration', fontsize=10)
        else:
            ax.tick_params(labelleft=False)
        
        # Add value annotations above each bar
        for bar, val in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.4f}',
                    ha='center', va='bottom', fontsize=8)

    #plt.savefig('./bayesian_optimisation_results_jsc.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("✓ Figures saved to: ./bayesian_optimisation_results_jsc.png")
else:
    print("⚠ No figures generated as no configurations improve upon the current maximum.")

In [ ]:
proposed_pce_data = pd.read_csv('./proposed_bo_experiments_pce.csv')
proposed_voc_data = pd.read_csv('./proposed_bo_experiments_voc.csv')
proposed_jsc_data = pd.read_csv('./proposed_bo_experiments_jsc.csv')


In [ ]:

g = sns.pairplot(proposed_pce_data.drop(columns=['Experiment', 'Improvement(%)']), diag_kind='kde')

# Iterate over all subplots to adjust font sizes
for ax in g.axes.flatten():
    if ax:
        # Increase axis label sizes (x and y)
        ax.set_xlabel(ax.get_xlabel(), fontsize=20)
        ax.set_ylabel(ax.get_ylabel(), fontsize=20)
        
        # Increase tick label sizes (numbers on axes)
        ax.tick_params(axis='both', labelsize=16)

g.fig.suptitle('Pairplot - Proposed points', y=1.02, fontsize=20)
plt.show()


In [ ]:
fig=plt.figure(figsize=(10, 10))
sns.pairplot(proposed_voc_data.drop(columns=['Experiment', 'Improvement(V)']), diag_kind='kde')
plt.suptitle('Pairplot – Proposed Bayesian Optimisation Points Voc(V)', y=1.02)
plt.show()

In [ ]:
fig=plt.figure(figsize=(10, 10))
sns.pairplot(proposed_jsc_data.drop(columns=['Experiment', 'Improvement(uAcm-2)']), diag_kind='kde')
plt.suptitle('Pairplot – Proposed Bayesian Optimisation Points Jsc(uAcm-2)', y=1.02)
plt.show()

In [ ]:
# Load experimental data
df_exp = pd.read_csv('./data-bo-dssc.tsv', sep='\t', decimal=',')

# Load optimisation proposals
df_pce = pd.read_csv('./proposed_bo_experiments_pce.csv')
df_jsc = pd.read_csv('./proposed_bo_experiments_jsc.csv')
df_voc = pd.read_csv('./proposed_bo_experiments_voc.csv')

# Input components and target variables
components = ['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']
targets = ['PCE', 'Jsc', 'Voc']
propuestas = {'PCE': df_pce, 'Jsc': df_jsc, 'Voc': df_voc}

# Create figure: 3 rows × 5 columns
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

for i, target in enumerate(targets):
    for j, comp in enumerate(components):
        ax = axes[i, j]
        
        # Experimental data
        exp_data = df_exp[comp].values
        
        # Proposed data
        prop_data = propuestas[target][comp].values
        
        # Overlaid histograms
        ax.hist(exp_data, bins=15, alpha=0.6, label='Experimental', color='blue', density=True)
        ax.hist(prop_data, bins=15, alpha=0.6, label='Proposed', color='orange', density=True)
        
        # Axis labels
        if i == 0:
            ax.set_title(comp, fontsize=14, fontweight='bold')
        if j == 0:
            ax.set_ylabel(target, fontsize=14, fontweight='bold')
        
        # Legend in the first cell only
        if i == 0 and j == 0:
            ax.legend(fontsize=10)

plt.tight_layout()
#plt.savefig('histograms_experimental_vs_proposed.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Surrogate Model Validation — Leave-One-Out Cross-Validation (LOOCV)

Before trusting the optimiser's recommendations, it is important to assess how well the Gaussian Process surrogate actually predicts experimental outcomes.

**Leave-One-Out Cross-Validation (LOOCV)** provides a rigorous estimate of predictive accuracy without requiring a separate test set — especially valuable when data are scarce. In each fold, one experiment is held out, the GP is re-fitted on the remaining data, and a prediction is made for the held-out point. This is repeated for every experiment, yielding a set of *predicted vs. measured* pairs that quantify the model's generalisation ability.

The same Matérn kernel used during optimisation is employed here, ensuring that the validation reflects the actual surrogate quality.

In [ ]:
# Reload the full dataset
df_loocv = pd.read_csv('./data-bo-dssc.tsv', sep='\t', decimal=',')
scaler = StandardScaler()
X_all = df_loocv[['I2', 'LiI', 'BMII', 'TBP', 'GuSCN']].values
X_all = scaler.fit_transform(X_all)

targets_loocv = {
    'PCE (%)':        df_loocv['PCE(%)'].values,
    'Voc (V)':        df_loocv['Voc(V)'].values,
    'Jsc (µA cm⁻²)': df_loocv['Jsc(uAcm-2)'].values,
}

# Per-target kernels, matching those used during optimisation
kernels_loocv = {
    'PCE (%)': (
        ConstantKernel(3.5, (1e-3, 1e1))
        * Matern(length_scale=3 * np.ones(5), length_scale_bounds=(1e-4, 1e1), nu=2.5)
        + WhiteKernel(noise_level=0.3, noise_level_bounds=(1e-6, 1e-3))
    ),
    'Voc (V)': (
        ConstantKernel(1e-3, (1e-5, 1e-1))
        * Matern(length_scale=np.ones(5), length_scale_bounds=(1e-2, 5*1e1), nu=2.5)
        + WhiteKernel(noise_level=1e-4, noise_level_bounds=(1e-6, 1e-1))
    ),
    'Jsc (µA cm⁻²)': (
        ConstantKernel(1e2, (1e-1, 1e3))
        * Matern(length_scale=np.ones(5), length_scale_bounds=(1e-2, 1e1), nu=2.5)
        + WhiteKernel(noise_level=1e0, noise_level_bounds=(1e-6, 1e1))
    ),
}

loo = LeaveOneOut()
loocv_results = {}

for target_name, y_all in targets_loocv.items():
    y_true, y_pred_loo, y_std_loo = [], [], []
    kernel = kernels_loocv[target_name]

    for train_idx, test_idx in loo.split(X_all):
        X_train, X_test = X_all[train_idx], X_all[test_idx]
        y_train, y_test = y_all[train_idx], y_all[test_idx]

        gp = GaussianProcessRegressor(
            kernel=kernel,
            n_restarts_optimizer=20,
            random_state=42,
            normalize_y=False
        )
        gp.fit(X_train, y_train)
        pred, std = gp.predict(X_test, return_std=True)

        y_true.append(y_test[0])
        y_pred_loo.append(pred[0])
        y_std_loo.append(std[0])
        print(gp.log_marginal_likelihood_value_)

    y_true = np.array(y_true)
    y_pred_loo = np.array(y_pred_loo)
    y_std_loo = np.array(y_std_loo)
    loocv_results[target_name] = (y_true, y_pred_loo, y_std_loo)

    r2 = np.corrcoef(y_true, y_pred_loo)[0, 1] ** 2
    mae = mean_absolute_error(y_true, y_pred_loo)
    print(f"{target_name:20s}  R² = {r2:.4f}   MAE = {mae:.4f}")
    print(f"Optimised kernel for {target_name}: {gp.kernel_}\n")

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (target_name, (y_true, y_pred_loo, y_std_loo)) in zip(axes, loocv_results.items()):
    if target_name == 'PCE (%)':
        results = pd.DataFrame({
            'True': y_true,
            'Predicted': y_pred_loo,
            'Std': y_std_loo
        })
        results.to_csv(f'./results-loocv-{target_name}.csv', index=False)
    r2  = r2_score(y_true, y_pred_loo)
    mae = mean_absolute_error(y_true, y_pred_loo)

    lims = [min(y_true.min(), (y_pred_loo - y_std_loo).min()) * 0.95,
            max(y_true.max(), (y_pred_loo + y_std_loo).max()) * 1.05]

    ax.errorbar(
        y_true, y_pred_loo,
        yerr=y_std_loo,
        fmt='o', alpha=0.7,
        ecolor='steelblue', elinewidth=1.2, capsize=3,
        markeredgecolor='k', markeredgewidth=0.5,
        zorder=3, label='LOO prediction ± 1σ'
    )
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel(f'Measured {target_name}', fontsize=12)
    ax.set_ylabel(f'GP predicted {target_name}', fontsize=12)
    ax.set_title(f'{target_name}\n$R^2$ = {r2:.3f}   MAE = {mae:.3f}', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')

plt.suptitle('Gaussian Process Surrogate — LOOCV Validation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()